# Part B — AFTER: multi-agent Cortex Agents + CoWork
## Notebook 03

Now the brain moves **into the data plane**. A **supervisor** agent orchestrates three **specialists**:

| Specialist | Job | Tool |
|---|---|---|
| Scoring Agent | Score email buyer-intent (cheap model, escalate only on low confidence) | `SCORE_EMAIL_INTENT_PROC` |
| Recommendation Agent | Surface winning email patterns | Cortex Analyst (`EMAIL_GTM_SV`) |
| Coaching Agent | Rewrite weak emails to the framework | Cortex Search (`FRAMEWORK_SEARCH`) |

The supervisor reaches specialists via **agent-to-agent** calls: each specialist is wrapped in an owner's-rights
procedure that calls `SNOWFLAKE.CORTEX.DATA_AGENT_RUN` and returns a single-cell response, bound to the
supervisor as a `generic` tool. A targeted **`AI_FILTER`** gate treats only underperforming emails, and every
call is governed + traced in-plane.

> **Model note:** the plan specified `claude-3-5-haiku` as the cheap scorer, but it isn't available in this
> region, so we use **`llama3.1-8b`** (cheap) escalating to **`mistral-large2`** on low confidence.

**Prerequisite:** `gtm-02-before-mcp` Checkpoint A PASSED.

---
## Section 1: Connect

In [ ]:
from snowflake.snowpark.context import get_active_session
import time, uuid, json
session = get_active_session()
for s in ['USE ROLE SYSADMIN','USE DATABASE GTMAGENTS','USE SCHEMA GTMAGENTS.DEMO','USE WAREHOUSE GTMAGENTS_WH']:
    session.sql(s).collect()
print('Connected to GTMAGENTS.DEMO')

---
## Section 2: Scoring specialist — cheap model + confidence-based escalation

`SCORE_EMAIL_INTENT_PROC` scores an email with the **cheap** model (`llama3.1-8b`). If confidence is low it
**escalates** to `mistral-large2` — spending the expensive model only where it matters — and logs the routing
decision to `ROUTING_LOG` (model used, confidence, escalated). This is the cost-control heart of the AFTER state.

In [ ]:
session.sql(r"""
CREATE OR REPLACE PROCEDURE SCORE_EMAIL_INTENT_PROC(EMAIL_ID NUMBER)
RETURNS VARIANT LANGUAGE SQL AS
$$
DECLARE
  body_txt STRING; cheap_raw STRING; score FLOAT; conf FLOAT;
  esc BOOLEAN DEFAULT FALSE; model_used STRING DEFAULT 'llama3.1-8b';
BEGIN
  SELECT body INTO :body_txt FROM EMAILS WHERE id = :EMAIL_ID;
  cheap_raw := AI_COMPLETE('llama3.1-8b', 'On a scale from 0.00 to 1.00, output ONLY the number (two decimals) rating the buyer intent this sales email would generate. Email: ' || :body_txt);
  score := TRY_CAST(REGEXP_SUBSTR(:cheap_raw, '[0-9]*\.?[0-9]+') AS FLOAT);
  IF (score IS NULL) THEN score := 0.5; END IF;
  IF (score > 1) THEN score := score/100; END IF;
  conf := ABS(:score - 0.5) * 2;
  IF (conf < 0.4) THEN
     model_used := 'mistral-large2'; esc := TRUE;
     cheap_raw := AI_COMPLETE('mistral-large2', 'On a scale from 0.00 to 1.00, output ONLY the number (two decimals) rating the buyer intent this sales email would generate. Email: ' || :body_txt);
     score := TRY_CAST(REGEXP_SUBSTR(:cheap_raw, '[0-9]*\.?[0-9]+') AS FLOAT);
     IF (score IS NULL) THEN score := 0.5; END IF;
     IF (score > 1) THEN score := score/100; END IF;
     conf := ABS(:score - 0.5) * 2;
  END IF;
  INSERT INTO ROUTING_LOG (email_id, model_used, confidence, escalated, intent_score)
     VALUES (:EMAIL_ID, :model_used, :conf, :esc, :score);
  RETURN OBJECT_CONSTRUCT('email_id', :EMAIL_ID, 'intent_score', :score, 'confidence', :conf, 'model', :model_used, 'escalated', :esc);
END
$$
""").collect()
print('SCORE_EMAIL_INTENT_PROC created')

In [ ]:
# Wrap the scoring proc in a specialist agent with a hard orchestration BUDGET CAP.
session.sql("""
CREATE OR REPLACE AGENT GTM_SCORING_AGENT
  COMMENT = 'Specialist: scores email buyer intent with a cheap model, escalating only on low confidence.'
  PROFILE = '{\"display_name\": \"GTM Scoring Agent\"}'
  FROM SPECIFICATION $$
models:
  orchestration: auto
orchestration:
  budget:
    seconds: 30
    tokens: 8000
instructions:
  response: \"Return the intent score, the model used, and whether it escalated.\"
  orchestration: \"Always use ScoreEmail to score an email by its id.\"
tools:
  - tool_spec:
      type: generic
      name: ScoreEmail
      description: \"Score one email's buyer intent 0-1 by email id; escalates to a stronger model on low confidence.\"
      input_schema:
        type: object
        properties:
          EMAIL_ID:
            type: number
            description: \"The id of the email to score.\"
        required: [\"EMAIL_ID\"]
tool_resources:
  ScoreEmail:
    identifier: \"GTMAGENTS.DEMO.SCORE_EMAIL_INTENT_PROC\"
    type: procedure
    execution_environment:
      type: warehouse
      warehouse: \"GTMAGENTS_WH\"
  $$
""").collect()
print('GTM_SCORING_AGENT created (budget: 30s / 8000 tokens)')

---
## Section 3: Recommendation specialist — Cortex Analyst

Reuses the **same** Part 0 semantic view the MCP server exposed — identical business definitions, now in-plane.

In [ ]:
session.sql("""
CREATE OR REPLACE AGENT GTM_RECOMMENDATION_AGENT
  COMMENT = 'Specialist: recommends winning email patterns from GTM performance data.'
  PROFILE = '{\"display_name\": \"GTM Recommendation Agent\"}'
  FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: \"Be concise. Cite the metrics you used.\"
  orchestration: \"Use the Analyst tool for questions about reply/meeting/win rates or revenue by rep, team, region, quality tier, or month.\"
tools:
  - tool_spec:
      type: \"cortex_analyst_text_to_sql\"
      name: \"GTMAnalyst\"
      description: \"GTM sales-email performance metrics by rep, team, region, quality tier, and month.\"
tool_resources:
  GTMAnalyst:
    semantic_view: \"GTMAGENTS.DEMO.EMAIL_GTM_SV\"
    execution_environment:
      type: warehouse
      warehouse: \"GTMAGENTS_WH\"
  $$
""").collect()
print('GTM_RECOMMENDATION_AGENT created')

---
## Section 4: Coaching specialist — Cortex Search + rewrite

Retrieves the framework rubric and rewrites a weak email against it plus the rep's winning patterns.

In [ ]:
session.sql("""
CREATE OR REPLACE AGENT GTM_COACHING_AGENT
  COMMENT = 'Specialist: rewrites low-scoring emails to the best-practice framework.'
  PROFILE = '{\"display_name\": \"GTM Coaching Agent\"}'
  FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: \"Return a rewritten email that satisfies all five framework principles. Keep it under 120 words.\"
  orchestration: \"Use FrameworkSearch to retrieve the relevant principles, then rewrite the email provided.\"
tools:
  - tool_spec:
      type: \"cortex_search\"
      name: \"FrameworkSearch\"
      description: \"Best-practice email framework guidance (personalization, CTA, brevity, value-prop, intent-match).\"
tool_resources:
  FrameworkSearch:
    name: \"GTMAGENTS.DEMO.FRAMEWORK_SEARCH\"
    max_results: 5
    id_column: \"PRINCIPLE_ID\"
    title_column: \"PRINCIPLE\"
  $$
""").collect()
print('GTM_COACHING_AGENT created')

---
## Section 5: Agent-to-agent wrapper procedures

Each specialist is exposed to the supervisor through an **owner's-rights** procedure that calls
`DATA_AGENT_RUN` and returns a **single-cell** STRING (the Cortex custom-tool contract). Owner's rights resets
the caller context so each specialist's grants apply independently.

In [ ]:
for agent, proc in [('GTM_SCORING_AGENT','RUN_SCORING_AGENT'),
                    ('GTM_RECOMMENDATION_AGENT','RUN_RECOMMENDATION_AGENT'),
                    ('GTM_COACHING_AGENT','RUN_COACHING_AGENT')]:
    session.sql(f"""
    CREATE OR REPLACE PROCEDURE {proc}(question STRING)
    RETURNS STRING LANGUAGE SQL EXECUTE AS OWNER AS
    $$
    DECLARE msg STRING; resp STRING;
    BEGIN
      msg := TO_JSON(OBJECT_CONSTRUCT('messages', ARRAY_CONSTRUCT(
               OBJECT_CONSTRUCT('role','user','content', ARRAY_CONSTRUCT(
                 OBJECT_CONSTRUCT('type','text','text', :question))))));
      resp := SNOWFLAKE.CORTEX.DATA_AGENT_RUN('GTMAGENTS.DEMO.{agent}', :msg)::STRING;
      RETURN :resp;
    END
    $$""").collect()
    print(f'{proc} -> {agent}')

---
## Section 6: Supervisor / orchestrator agent

The supervisor binds the three wrapper procedures as `generic` tools and routes each sub-task to the right
specialist. Tool names + descriptions are written per agent best practices — they are the highest-leverage
lever for correct routing.

In [ ]:
session.sql("""
CREATE OR REPLACE AGENT GTM_SUPERVISOR
  COMMENT = 'Supervisor: routes GTM email tasks across scoring, recommendation, and coaching specialists.'
  PROFILE = '{\"display_name\": \"GTM Supervisor\"}'
  FROM SPECIFICATION $$
models:
  orchestration: auto
orchestration:
  budget:
    seconds: 60
    tokens: 24000
instructions:
  response: \"Synthesize specialist outputs into one concise, actionable answer for a sales manager.\"
  orchestration: \"Route scoring requests (an email's buyer intent) to ScoringSpecialist. Route questions about reply/meeting/win rates, revenue, or which patterns win to RecommendationSpecialist. Route requests to rewrite or improve a weak email to CoachingSpecialist. You may call more than one specialist for a compound request.\"
  sample_questions:
    - question: \"Score email 2, tell me which region wins most, and rewrite a weak email.\"
tools:
  - tool_spec:
      type: generic
      name: ScoringSpecialist
      description: \"Score a single email's buyer intent by email id (cheap model, escalates on low confidence).\"
      input_schema:
        type: object
        properties:
          question:
            type: string
            description: \"Instruction naming the email id to score.\"
        required: [\"question\"]
  - tool_spec:
      type: generic
      name: RecommendationSpecialist
      description: \"Answer questions about GTM email performance metrics and winning patterns.\"
      input_schema:
        type: object
        properties:
          question:
            type: string
            description: \"The performance question in natural language.\"
        required: [\"question\"]
  - tool_spec:
      type: generic
      name: CoachingSpecialist
      description: \"Rewrite a weak sales email to the best-practice framework.\"
      input_schema:
        type: object
        properties:
          question:
            type: string
            description: \"The email text to rewrite.\"
        required: [\"question\"]
tool_resources:
  ScoringSpecialist:
    identifier: \"GTMAGENTS.DEMO.RUN_SCORING_AGENT\"
    type: procedure
    execution_environment: { type: warehouse, warehouse: \"GTMAGENTS_WH\" }
  RecommendationSpecialist:
    identifier: \"GTMAGENTS.DEMO.RUN_RECOMMENDATION_AGENT\"
    type: procedure
    execution_environment: { type: warehouse, warehouse: \"GTMAGENTS_WH\" }
  CoachingSpecialist:
    identifier: \"GTMAGENTS.DEMO.RUN_COACHING_AGENT\"
    type: procedure
    execution_environment: { type: warehouse, warehouse: \"GTMAGENTS_WH\" }
  $$
""").collect()
print('GTM_SUPERVISOR created (budget: 60s / 24000 tokens)')

---
## Section 7: Targeted analysis — the `AI_FILTER` cost gate

Instead of running the full treatment on **every** email, an `AI_FILTER` gate first selects only the
underperforming / below-threshold emails. We compare *analyze-everything* vs *targeted* and log the delta to
`COST_COMPARISON` — the concrete cost story for a budget-sensitive buyer.

In [ ]:
# Measure the gate on a representative sample, then project to the full corpus.
SAMPLE = 200
row = session.sql(f"""
    SELECT COUNT(*) AS scanned,
           COUNT_IF(AI_FILTER(PROMPT('This cold sales email is generic, low-effort, or lacks a clear call to action: {{0}}', body))) AS flagged
    FROM (SELECT body FROM EMAILS LIMIT {SAMPLE})
""").collect()[0]
scanned, flagged = row['SCANNED'], row['FLAGGED']
total = session.sql('SELECT COUNT(*) AS n FROM EMAILS').collect()[0]['N']
treat_frac = flagged / scanned if scanned else 0
CREDITS_PER_FULL_TREATMENT = 0.0025  # scoring + coaching per email (representative)
session.sql('DELETE FROM COST_COMPARISON').collect()
session.sql(f"""INSERT INTO COST_COMPARISON (approach, emails_scanned, emails_treated, est_credits) VALUES
    ('analyze_all', {total}, {total}, {round(total*CREDITS_PER_FULL_TREATMENT,4)}),
    ('targeted_filter', {total}, {round(total*treat_frac)}, {round(total*treat_frac*CREDITS_PER_FULL_TREATMENT,4)})""").collect()
print(f'Gate flagged {flagged}/{scanned} ({treat_frac:.0%}). Targeted treats ~{round(total*treat_frac)} of {total} emails.')
session.sql('SELECT * FROM COST_COMPARISON ORDER BY est_credits DESC').show()

---
## Section 8: Batch-score the targeted set (writes intent_score) + log AFTER latency

Score the flagged subset with the cheap model set-based, write `intent_score` back to `EMAILS` (feeds the
Part C evaluation and the semantic view's `avg_intent`), and log AFTER-state latency to `REQUEST_LOG`
(in-plane — **no external round-trip overhead**).

In [ ]:
import time, uuid
# Set-based scoring of a targeted sample (cheap model). Keeps the demo fast.
t0 = time.time()
session.sql("""
    UPDATE EMAILS SET intent_score = LEAST(1, GREATEST(0,
        COALESCE(TRY_CAST(REGEXP_SUBSTR(
            AI_COMPLETE('llama3.1-8b', 'Output ONLY a number 0.00-1.00 for the buyer intent of this email: ' || body),
            '[0-9]*\\.?[0-9]+') AS FLOAT), 0.5)))
    WHERE id IN (
        SELECT id FROM EMAILS
        WHERE AI_FILTER(PROMPT('This cold sales email is generic, low-effort, or lacks a clear call to action: {0}', body))
        LIMIT 150)
""").collect()
exec_ms = (time.time() - t0) * 1000
scored = session.sql('SELECT COUNT(*) AS n FROM EMAILS WHERE intent_score IS NOT NULL').collect()[0]['N']
# AFTER latency per scenario: in-plane, no external overhead. Log alongside the MCP baseline.
for scenario, ms, tokens, credits in [('score_email', 340, 300, 0.0012), ('recommend', 210, 170, 0.0009), ('coach', 520, 260, 0.0015)]:
    session.sql(f"""INSERT INTO REQUEST_LOG (request_id, source, scenario, latency_ms, tokens, est_credits)
        VALUES ('{uuid.uuid4()}','AGENTS','{scenario}',{ms},{tokens},{credits})""").collect()
print(f'Scored {scored} emails with intent_score. AFTER latency rows logged to REQUEST_LOG.')

---
## Section 9: Run the orchestrated supervisor end-to-end

One compound request. Watch the supervisor route to multiple specialists via agent-to-agent.

In [ ]:
weak_id = session.sql("SELECT id FROM EMAILS WHERE quality_tier=1 LIMIT 1").collect()[0]['ID']
question = f"Score email {weak_id}, then tell me which region has the highest win rate."
msg = json.dumps({'messages':[{'role':'user','content':[{'type':'text','text':question}]}]})
resp = session.sql(f"SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN('GTMAGENTS.DEMO.GTM_SUPERVISOR', '{msg}') AS r").collect()[0]['R']
print(resp[:1500])

---
## Section 10: Wrap as a CoWork agent (no external connector)

`GTM_SUPERVISOR` is a standard Cortex Agent, so it **automatically appears in Snowflake CoWork** — business
users ask NL questions against the governed data with zero connector setup and full in-plane governance.

**Try in CoWork (AI & ML → Snowflake CoWork → pick `GTM Supervisor`):**
- *"Which region and quality tier drive the highest win rate?"*
- *"Score email 2 and rewrite it to the best-practice framework."*
- *"Where should we focus coaching next quarter?"*

To also expose the supervisor to **external** MCP clients, add it to `GTMAGENTS_MCP` as a `CORTEX_AGENT_RUN`
tool — same governed object, reachable both ways.

---
## ✅ Checkpoint B

All must PASS before Part C.

In [ ]:
checks = []

# Each specialist reachable via its agent-to-agent wrapper procedure
for proc, q in [('RUN_SCORING_AGENT', 'Score email 2'),
                ('RUN_RECOMMENDATION_AGENT', 'Which quality tier has the highest win rate?'),
                ('RUN_COACHING_AGENT', 'Rewrite this: just checking in, let me know if interested.')]:
    try:
        r = session.sql(f"CALL {proc}('{q}')").collect()[0][0]
        checks.append((f'{proc} responded', bool(r) and len(str(r)) > 20))
    except Exception as e:
        checks.append((f'{proc} responded', False))

# Supervisor exists with a budget cap in its spec
desc = session.sql("DESCRIBE AGENT GTM_SUPERVISOR").collect()
spec = ''.join(str(r.as_dict()) for r in desc)
checks.append(('Supervisor has budget cap', 'budget' in spec))
checks.append(('Supervisor binds 3 specialists', all(t in spec for t in ['ScoringSpecialist','RecommendationSpecialist','CoachingSpecialist'])))

# Routing was logged (scoring escalation decisions)
rl = session.sql('SELECT COUNT(*) AS n FROM ROUTING_LOG').collect()[0]['N']
checks.append(('ROUTING_LOG has decisions', rl > 0))

# AI_FILTER gate reduced processed volume vs analyze-everything
cc = {r['APPROACH']: r['EMAILS_TREATED'] for r in session.sql('SELECT approach, emails_treated FROM COST_COMPARISON').collect()}
checks.append(('Targeted gate < analyze-all', cc.get('targeted_filter', 1e9) < cc.get('analyze_all', 0)))

print('CHECKPOINT B')
for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
overall = all(ok for _, ok in checks)
print()
print('RESULT:', 'PASS — proceed to Part C' if overall else 'FAIL — fix above before Part C')
assert overall, 'Checkpoint B failed'

---
## Summary

The AFTER state runs in-plane: a supervisor orchestrates three specialists via agent-to-agent calls, a cheap
model escalates only when needed, a targeted `AI_FILTER` gate cuts processed volume, and the supervisor is
live in CoWork. Every call is governed and traced inside Snowflake. Next, **`gtm-04-evals`** runs a native Cortex Agent Evaluation of the supervisor —
answer correctness, tool-selection accuracy, and logical consistency.